# Importing modules


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes

# Loading Data


In [2]:
data = load_diabetes()
X,y = data.data, data.target

# Feature Scaling


In [3]:
X_mean, X_std = np.mean(X, axis = 0), np.std(X, axis = 0)
y_mean, y_std = np.mean(y, axis = 0), np.std(y, axis = 0)
X = (X - X_mean)/X_std
y = (y - y_mean)/y_std
print(X.shape, y.shape)

(442, 10) (442,)


# Break Data into train, validation and test data

In [4]:
from sklearn.model_selection import train_test_split
train_X, test_X, train_y, test_y = train_test_split(X,y,test_size=0.2,random_state=0)
train_X, val_X, train_y, val_y = train_test_split(train_X,train_y,test_size=0.2,random_state=0)

In [5]:
print(train_X.shape, train_y.shape)
print(val_X.shape, val_y.shape)
print(test_X.shape, test_y.shape)

(282, 10) (282,)
(71, 10) (71,)
(89, 10) (89,)


# RBF Kernel function



In [6]:
def K(X1,X2,gamma=0.1): #RBF Kernel function
  sq_dist = np.sum(X1**2,axis = 1).reshape(-1,1) + np.sum(X2**2,axis=1).reshape(1,-1) - 2*(X1@X2.T) #Mistake 1: Forgot reshape(-1,1) and (1,-1)
  return np.exp(-gamma*sq_dist)

# Kernelised Primal model training, validation and testing

In [7]:
alpha = np.zeros(train_X.shape[0]) #mistake 2: I did X.shape[1]
alpha_star = np.zeros(train_X.shape[0])
beta = alpha - alpha_star
b = 0

epochs = 2000
epsilon = 1e-5
C = 1
lr = 3e-5

K_train = K(train_X,train_X)
K_val = K(val_X,train_X)
K_test = K(test_X,train_X)
print(K_train.shape, K_val.shape, K_test.shape,beta.shape)

(282, 282) (71, 282) (89, 282) (282,)


In [8]:
for epoch in range(epochs):
  train_y_pred = K_train@beta + b
  train_xi = np.maximum(0, train_y_pred - train_y - epsilon)
  train_xi_star = np.maximum(0, train_y - train_y_pred - epsilon)
  w_norm_sq = beta.T@K_train@beta
  train_J = 0.5 * w_norm_sq + C*np.sum(train_xi + train_xi_star)
  residual = train_y_pred - train_y
  mask_pos = (residual > epsilon).astype(np.float64) #Mistake 3: wrote int instead of float64
  mask_neg = (residual < -epsilon).astype(np.float64)
  dJ_dbeta = K_train @ beta + C * (K_train@(mask_pos - mask_neg).T)
  dJ_db = C * np.sum(mask_pos - mask_neg)
  beta -= lr*dJ_dbeta
  b -= lr*dJ_db

  if (epoch+1)%100==0:
    val_y_pred = K_val@beta + b
    mseloss = np.mean((val_y_pred - val_y)**2)
    print(f"Epoch {epoch+1}: val loss = {mseloss}")

Epoch 100: val loss = 0.841768276040152
Epoch 200: val loss = 0.723588036335928
Epoch 300: val loss = 0.6707952883485186
Epoch 400: val loss = 0.6504875102051157
Epoch 500: val loss = 0.6327724970415736
Epoch 600: val loss = 0.6184123388084586
Epoch 700: val loss = 0.6149328625786902
Epoch 800: val loss = 0.6149827795149466
Epoch 900: val loss = 0.610947025588567
Epoch 1000: val loss = 0.6034911897607252
Epoch 1100: val loss = 0.5989000400886009
Epoch 1200: val loss = 0.5941859663068589
Epoch 1300: val loss = 0.5909201939050397
Epoch 1400: val loss = 0.5864617651496467
Epoch 1500: val loss = 0.5828698123120336
Epoch 1600: val loss = 0.5796327938323826
Epoch 1700: val loss = 0.575544043277097
Epoch 1800: val loss = 0.5737515135038757
Epoch 1900: val loss = 0.5699189850627193
Epoch 2000: val loss = 0.5674400537500651


In [9]:
test_y_pred = K_test@beta + b
mseloss = np.mean((test_y_pred - test_y)**2)
print(f"test loss = {mseloss}")

test loss = 0.5866862453581726


# Kernelised Dual model training, validation and testing

In [10]:
alpha = np.zeros(train_X.shape[0]) #mistake 2: I did X.shape[1]
alpha_star = np.zeros(train_X.shape[0])
beta = alpha - alpha_star
b = 0

epochs = 2000
epsilon = 1e-5
C = 1
lr = 3e-5

K_train = K(train_X,train_X)
K_val = K(val_X,train_X)
K_test = K(test_X,train_X)
print(K_train.shape, K_val.shape, K_test.shape,beta.shape)

(282, 282) (71, 282) (89, 282) (282,)


In [11]:
for epoch in range(epochs):
  train_y_pred = K_train@beta + b
  train_W = -0.5 * (beta.T@K_train@beta) - epsilon * np.sum(alpha + alpha_star) + np.sum(train_y * beta) #must be maximized
  dW_dalpha = - K_train@beta - epsilon + train_y
  dW_dalpha_star = K_train@beta - epsilon - train_y
  alpha += lr*dW_dalpha
  alpha_star += lr*dW_dalpha_star
  alpha -= 0.5* np.sum(alpha-alpha_star)/len(alpha) # to make np.sum(alpha - alpha*) = 0
  alpha_star += 0.5* np.sum(alpha-alpha_star)/len(alpha_star)# to make np.sum(alpha - alpha*) = 0
  alpha = np.clip(alpha,0,C)
  alpha_star = np.clip(alpha_star,0,C)
  beta = alpha - alpha_star
  b = np.mean(- K_train@beta - epsilon + train_y)

  if (epoch+1)%100==0:
    val_y_pred = K_val@beta + b
    mseloss = np.mean((val_y_pred - val_y)**2)
    print(f"Epoch {epoch+1}: val loss = {mseloss}")

Epoch 100: val loss = 1.075924119597794
Epoch 200: val loss = 1.0241331973379995
Epoch 300: val loss = 0.9792024397622672
Epoch 400: val loss = 0.9400770374926037
Epoch 500: val loss = 0.9059541038714701
Epoch 600: val loss = 0.8761061749528725
Epoch 700: val loss = 0.8499138496110138
Epoch 800: val loss = 0.826811582551119
Epoch 900: val loss = 0.8063201107474041
Epoch 1000: val loss = 0.788151870468118
Epoch 1100: val loss = 0.771978673242237
Epoch 1200: val loss = 0.757542968413811
Epoch 1300: val loss = 0.7446151369778052
Epoch 1400: val loss = 0.7329479503248117
Epoch 1500: val loss = 0.7224089193273451
Epoch 1600: val loss = 0.7128499529023251
Epoch 1700: val loss = 0.7041348464215453
Epoch 1800: val loss = 0.6961968776404526
Epoch 1900: val loss = 0.6889438273837866
Epoch 2000: val loss = 0.6823201340194521


In [12]:
test_y_pred = K_test@beta + b
mseloss = np.mean((test_y_pred - test_y)**2)
print(f"test loss = {mseloss}")

test loss = 0.5674503539873651
